## 1. Setup

### TreeSHAP — Rationale for Exact, Fast Explanations

SHAP (SHapley Additive exPlanations) decomposes each model prediction into
**additive contributions from individual features**, grounded in cooperative
game theory. For our LightGBM champion model we use **TreeSHAP** (Lundberg et al., 2018),
which computes exact SHAP values in **polynomial time** by exploiting the tree structure —
unlike KernelSHAP (model-agnostic) which uses Monte Carlo approximations.

Key properties of TreeSHAP:
- **Efficiency:** O(TLD²) where T = trees, L = leaves, D = depth — milliseconds per sample.
- **Consistency:** larger SHAP magnitude always means larger marginal contribution.
- **Local accuracy:** SHAP values sum to the difference between predicted log-odds
  and the expected log-odds across all training samples.

SHAP values enable three types of explanation used in this notebook:
1. **Global importance** — which features drive predictions across all patients.
2. **Dependence plots** — how a feature's value affects its SHAP contribution.
3. **Local (per-patient) explanations** — why a specific patient is high-risk.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src.models import load_model
from src.preprocessing import split_data, scale_numeric
from src.explainability import (
    compute_shap_values,
    plot_global_summary,
    plot_bar_importance,
    plot_dependence,
    explain_patient
)

os.makedirs('../reports/figures', exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load champion model
model = load_model('lgbm_best')
print(f'Model loaded: {type(model).__name__}')

# Load and prepare test split
df = pd.read_csv('../data/processed/features_engineered.csv')
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df, target='hospital_death')
X_train_s, X_val_s, X_test_s, scaler = scale_numeric(X_train, X_val, X_test)

print(f'Test set shape: {X_test_s.shape}')
print(f'Test positive rate: {y_test.mean():.4f}')

## 2. Global Feature Importance

Global importance is computed by averaging |SHAP| across all patients in the
test sample. We use 2,000 randomly sampled rows for speed — this is sufficient
for stable importance estimates on a dataset of this size.

Two complementary views:
- **Beeswarm / summary plot**: shows both magnitude and direction of each feature's
  effect. Red = high feature value, blue = low. Horizontal spread = importance.
- **Bar plot**: clean ranking by mean |SHAP| for reporting.

In [ ]:
# Sample 2000 rows from test set for speed
np.random.seed(42)
sample_idx = np.random.choice(len(X_test_s), size=min(2000, len(X_test_s)), replace=False)

if hasattr(X_test_s, 'iloc'):
    X_sample = X_test_s.iloc[sample_idx]
else:
    X_sample = X_test_s[sample_idx]

print('Computing SHAP values on 2,000-row sample...')
shap_values = compute_shap_values(model, X_sample)
print(f'SHAP values shape: {shap_values.values.shape}')

# Global beeswarm summary
print('\nPlotting global SHAP summary (beeswarm)...')
plot_global_summary(shap_values, X_sample, max_display=20,
                    save_path='../reports/figures/shap_global_summary.png')

# Bar importance plot
print('Plotting SHAP bar importance...')
plot_bar_importance(shap_values, max_display=20,
                    save_path='../reports/figures/shap_bar_importance.png')

# Print top-10 features by mean |SHAP|
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
feature_names = (
    X_sample.columns.tolist()
    if hasattr(X_sample, 'columns')
    else [f'feature_{i}' for i in range(X_sample.shape[1])]
)
shap_importance = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False)
print('\nTop 10 features by mean |SHAP|:')
print(shap_importance.head(10).round(4).to_string())

## 3. Dependence Plots for Top Features

SHAP dependence plots reveal the relationship between a feature's raw value and
its SHAP contribution to the prediction. They also visualise **interaction effects**
by colouring points by the value of the most interacting feature.

We examine:
- **`apache_2_diagnosis`**: the primary APACHE-II severity classification, expected
  to show a strong positive monotonic relationship with mortality SHAP contribution.
- **`age`**: expected to show increasing SHAP contribution with age, potentially
  with an acceleration above ~65-70 years.

In [ ]:
# Identify available top features from SHAP importance
top_features = shap_importance.head(10).index.tolist()

# Prefer 'apache_2_diagnosis' and 'age' if available, else use top 2 from SHAP
preferred = ['apache_2_diagnosis', 'age']
dep_features = [f for f in preferred if f in top_features]
if len(dep_features) < 2:
    dep_features = top_features[:2]

print(f'Dependence features: {dep_features}')

for feat in dep_features:
    save_path = f'../reports/figures/shap_dependence_{feat}.png'
    plot_dependence(shap_values, X_sample, feature=feat, save_path=save_path)
    print(f'Saved dependence plot for {feat} -> {save_path}')

## 4. Local Explanation — Highest-Risk Patient

A local explanation answers the question: *why did the model assign this specific
patient a high predicted probability?* This is clinically actionable — a clinician
can use the SHAP waterfall chart to understand the primary drivers of risk for an
individual patient and decide on targeted interventions.

We identify the test-set patient with the **highest predicted mortality probability**
and generate their local SHAP waterfall chart, showing how each feature pushes the
prediction above or below the baseline expected probability.

In [ ]:
# Get predicted probabilities for the full sample
y_pred_proba_sample = model.predict_proba(X_sample)[:, 1]
highest_risk_idx = np.argmax(y_pred_proba_sample)
highest_risk_prob = y_pred_proba_sample[highest_risk_idx]

if hasattr(X_sample, 'iloc'):
    patient_row = X_sample.iloc[[highest_risk_idx]]
else:
    patient_row = X_sample[[highest_risk_idx]]

print(f'Highest-risk patient index (in sample): {highest_risk_idx}')
print(f'Predicted mortality probability: {highest_risk_prob:.4f} ({highest_risk_prob*100:.1f}%%)')

# Get SHAP explanation for this patient
patient_shap = explain_patient(model, patient_row)

# Build waterfall bar chart manually for control over styling
patient_shap_vals = shap_values.values[highest_risk_idx]
feature_names_all = (
    X_sample.columns.tolist()
    if hasattr(X_sample, 'columns')
    else [f'feature_{i}' for i in range(X_sample.shape[1])]
)

# Top 15 by absolute SHAP for this patient
top_idx = np.argsort(np.abs(patient_shap_vals))[-15:][::-1]
top_names = [feature_names_all[i] for i in top_idx]
top_vals = patient_shap_vals[top_idx]

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#D65F5F' if v > 0 else '#4878CF' for v in top_vals]
y_pos = range(len(top_names))
ax.barh(list(y_pos), top_vals[::-1], color=colors[::-1], edgecolor='white', alpha=0.85)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP Value (impact on log-odds of mortality)')
ax.set_title(
    f'Local SHAP Explanation — Highest-Risk Patient\n'
    f'Predicted Probability: {highest_risk_prob:.3f} ({highest_risk_prob*100:.1f}%%)',
    fontweight='bold'
)

for i, (val, name) in enumerate(zip(top_vals[::-1], top_names[::-1])):
    ax.text(val + (0.001 if val >= 0 else -0.001), i,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/local_explanation_highest_risk.png', bbox_inches='tight')
plt.show()

## 5. UMAP Embedding of SHAP Values

While individual SHAP explanations are useful for single patients, **UMAP** applied
to the SHAP value matrix reveals **patient clusters with similar risk profiles** —
groups of patients whose predictions are driven by the same combination of features.

This is more informative than UMAP on raw features because the SHAP space is
**outcome-aligned**: patients near each other in SHAP space share similar
model-relevant risk patterns, not just similar demographic characteristics.
Colouring by predicted probability shows where high-risk clusters concentrate.

In [ ]:
import umap

shap_matrix = shap_values.values
print(f'Fitting UMAP on SHAP matrix of shape {shap_matrix.shape}...')

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.1,
    metric='euclidean',
    random_state=42,
    verbose=False
)
embedding = reducer.fit_transform(shap_matrix)
print(f'UMAP embedding shape: {embedding.shape}')

# Predicted probabilities for colouring
pred_probs = model.predict_proba(X_sample)[:, 1]

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    embedding[:, 0], embedding[:, 1],
    c=pred_probs,
    cmap='RdYlBu_r',
    s=8,
    alpha=0.65,
    vmin=0, vmax=1
)
cbar = plt.colorbar(scatter, ax=ax, shrink=0.85)
cbar.set_label('Predicted Mortality Probability', fontsize=10)
ax.set_xlabel('UMAP Dimension 1', fontsize=11)
ax.set_ylabel('UMAP Dimension 2', fontsize=11)
ax.set_title(
    'UMAP Embedding of SHAP Values\nColoured by Predicted Mortality Probability',
    fontsize=13, fontweight='bold'
)

# Annotate the highest-risk patient
ax.scatter(
    embedding[highest_risk_idx, 0], embedding[highest_risk_idx, 1],
    c='black', s=80, zorder=5, marker='*', label='Highest-risk patient'
)
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/umap_shap_embedding.png', bbox_inches='tight')
plt.show()

print(f'\nUMAP complete. High-risk cluster visible in the upper-right region (red points).')
print(f'Patients with similar SHAP patterns cluster together, enabling cohort-level insights.')